<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Evaluación en la Clasificación</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">La clasificación de texto necesita una evaluación que distinga bien tipos de acierto y error. Este notebook lleva la discusión general de métricas a un problema supervisado concreto.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Tareas como sentimiento, spam o temática pueden parecer simples, pero exigen leer con cuidado qué tipo de error comete el modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender cómo se evalúa un clasificador de texto y qué lectura conviene hacer de sus resultados.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Una puntuación global aceptable puede ocultar fallos relevantes en clases concretas.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Evaluar clasificación implica comparar etiquetas predichas con etiquetas reales mediante métricas apropiadas.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado con cabeza de clasificación y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. Su cabeza de clasificación se inicializa de nuevo y no se entrena en este ejemplo.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes fundamentales para calcular métricas y para ejecutar la evaluación en clasificación en NLP.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados para verificación.


In [ ]:
# Bloque 2 — Construcción del dataset de evaluación
"""Este bloque construye un dataset mínimo para evaluar el rendimiento del modelo en clasificación.
Este bloque importa "Dataset" de "datasets" para construir un dataset de evaluación mínimo.

Definimos un dataset mínimo con textos y etiquetas para la evaluación de modelos de clasificación de texto.
La variable "textos_eval" contiene una lista con los textos de entrada para la evaluación de modelos de clasificación de texto.
La variable "etiquetas_eval" contiene una lista de etiquetas correspondientes a cada uno de los textos de entrada.

Definimos el diccionario que contiene los textos y etiquetas a partir del cual se construye el dataset de evaluación.
La variable "diccionario_eval" almacena los textos y etiquetas que se utilizarán para construir el dataset de evaluación.

Definimos la lista "lista_columnas" que contiene los nombres de las columnas que se incluirán en el dataset tokenizado.
La variable "lista_columnas" contiene los nombres de las columnas que se incluirán en el dataset tokenizado.

Aplicamos la clase "Dataset", acompañada del método ".from_dict()", a la variable "dataset_eval" para crear el dataset de evaluación.
El argumento "diccionario_eval" es el diccionario definido previamente a partir del cual se construye el dataset de evaluación.
La variable "dataset_eval" contiene el dataset de evaluación sin tokenizar listo para ser tokenizado y utilizado en la evaluación 
de modelos de clasificación de texto. Sus cuatro ejemplos son ilustrativos y no permiten una evaluación representativa."""

from datasets import Dataset

textos_eval = ["El producto es excelente.", "La experiencia fue terrible.", "El envío llegó rápido.", "La calidad es deficiente."]
etiquetas_eval = [1, 0, 1, 0]
diccionario_eval = {"texto": textos_eval, "label": etiquetas_eval}
lista_columnas = ["input_ids", "attention_mask", "label"]

dataset_eval = Dataset.from_dict(diccionario_eval)

dataset_eval # Muestra el dataset de evaluación para verificación de su contenido.


In [ ]:
# Bloque 3 — Tokenización del dataset
"""Este bloque aplica tokenización al dataset de evaluación.
Definimos la función "def tokenize_batch(batch)" para tokenizar cada lote de textos del dataset de evaluación.
El parámetro "batch" es un diccionario que contiene los textos asociados a la clave "texto" del diccionario definido previamente.

Esta función retorna la llamada al tokenizador "tokenizador()" con los textos del lote, aplicando padding y truncado.
El argumento "batch["texto"]" contiene los textos del lote asociados a la clave "texto".
El parámetro "padding=" rellena las secuencias hasta la longitud de la más larga del lote.
El parámetro "truncation=" trunca las secuencias que exceden la longitud máxima del modelo.

Aplicamos el método ".map()" al dataset de evaluación para tokenizar sus textos.
El argumento "tokenize_batch" es la función que se aplica a cada lote para realizar la tokenización.
El parámetro "batched=" indica que la función se aplicará a lotes de ejemplos en lugar de a ejemplos individuales.
La variable "dataset_eval_tokenizado" almacena el dataset de evaluación con los campos necesarios para el modelo.

Aplicamos el método ".set_format()" al dataset de evaluación tokenizado para establecer el formato de salida.
El parámetro "type=" indica con el valor "torch" que los datos se devolverán como tensores de PyTorch.
El parámetro "columns=" especifica las columnas que se incluirán en la salida, en este caso, las columnas definidas en la lista "lista_columnas".

La variable "dataset_eval_tokenizado" almacena el dataset con los campos necesarios para el modelo.
Esta operación permite que el dataset tokenizado se pueda utilizar directamente en modelos de PyTorch para evaluación, 
entrenamiento o inferencia."""

def tokenize_batch(batch):
    return tokenizador(batch["texto"], padding=True, truncation=True)

dataset_eval_tokenizado = dataset_eval.map(tokenize_batch, batched=True)
dataset_eval_tokenizado.set_format(type="torch", columns=lista_columnas)

dataset_eval_tokenizado # Muestra el dataset de evaluación tokenizado con el formato establecido.


In [ ]:
# Bloque 4 — Definición de métricas para clasificación
"""Este bloque define las métricas fundamentales para evaluar modelos de clasificación.
Este bloque importa "numpy" para trabajar con números y realizar cálculos de métricas en Python.

Definimos la función "def compute_metrics(eval_pred)" para calcular las métricas de evaluación.
El parámetro "eval_pred" es una tupla que contiene los logits predichos por el modelo y las etiquetas reales.

Asignamos de forma múltiple los valores de la tupla "eval_pred" a las variables "logits" y "labels".
La variable "logits" contiene los valores predichos por el modelo antes de aplicar la función de activación.
La variable "labels" contiene las etiquetas reales correspondientes a las predicciones.

Aplicamos la función "np.argmax()" a los logits para obtener las clases predichas y las asignamos a la variable "predicciones".
El argumento "logits" indica la matriz de logits predichos por el modelo.
El argumento "axis=" indica con el valor "-1" que se debe tomar el índice del valor máximo a lo largo del último eje de la matriz de logits.
La variable "predicciones" contiene los índices de las clases con mayor logit, sin aplicar una función de activación.

Aplicamos el método ".mean()" a la comparación entre predicciones y etiquetas reales para calcular la exactitud (accuracy).
La expresión "(predicciones == labels)" genera un array booleano donde cada elemento indica si la predicción coincide con la etiqueta real.
La variable "accuracy" contiene el valor de exactitud calculado a partir de las predicciones y las etiquetas reales.

La variable "verdaderos_positivos" contiene el número de predicciones correctas de la clase positiva (1).
La variable "falsos_positivos" contiene el número de predicciones incorrectas de la clase positiva (1).
La variable "falsos_negativos" contiene el número de predicciones incorrectas de la clase negativa (0) cuando la etiqueta real es positiva (1).

La clase positiva o negativa se refiere a la etiqueta que representa el evento de interés (positiva) o su ausencia (negativa).
Estos recuentos se utilizan para calcular las métricas de la clase positiva; no se calcula un recuento separado de verdaderos negativos.

La variable "precision" contiene la precisión calculada como el cociente entre "verdaderos_positivos" y la suma de "verdaderos_positivos" y "falsos_positivos".
La variable "recall" contiene la exhaustividad calculada como el cociente entre "verdaderos_positivos" y la suma de "verdaderos_positivos" y "falsos_negativos".
La variable "f1" contiene la métrica F1 calculada a partir de la media armónica entre "precision" y "recall".
Se añade 1e-8 a los denominadores para evitar divisiones por cero, lo que también modifica ligeramente los resultados no nulos.

Esta función retorna un diccionario "{}" con las métricas calculadas con los datos de evaluación proporcionados.
Este diccionario permite acceder a cada métrica de manera individual mediante su clave correspondiente.

Asignamos la función "compute_metrics" a la variable "calcular_metricas" para que pueda ser utilizada posteriormente en la evaluación del modelo.
La variable "calcular_metricas" almacena una instancia de la función "compute_metrics()"."""

import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # Desempaquetamos las predicciones y las etiquetas reales.
    predicciones = np.argmax(logits, axis=-1)

    accuracy = (predicciones == labels).mean()

    verdaderos_positivos = ((predicciones == 1) & (labels == 1)).sum() # Positive prediction and positive label = True Positives (TP)
    falsos_positivos = ((predicciones == 1) & (labels == 0)).sum()     # Positive prediction and negative label = False Positives (FP)
    falsos_negativos = ((predicciones == 0) & (labels == 1)).sum()     # Negative prediction and positive label = False Negatives (FN)

    precision = verdaderos_positivos / (verdaderos_positivos + falsos_positivos + 1e-8)  # Se añade 1e-8 para evitar división por cero.
    recall = verdaderos_positivos / (verdaderos_positivos + falsos_negativos + 1e-8)     # Se añade 1e-8 para evitar división por cero.
    f1 = 2 * precision * recall / (precision + recall + 1e-8)                            # Se añade 1e-8 para evitar división por cero.

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

calcular_metricas = compute_metrics

calcular_metricas # Muestra una representación de la función compute_metrics para verificar su contenido.


In [ ]:
# Bloque 5 — Configuración de TrainingArguments
"""Este bloque define los parámetros de evaluación mediante TrainingArguments.
Este bloque importa "TrainingArguments" de "transformers" para definir los argumentos de entrenamiento.

Aplicamos la clase "TrainingArguments()" a la variable "argumentos" para definir los parámetros de entrenamiento.
El parámetro "output_dir=" especifica el directorio donde se guardarán los resultados de la clasificación, en este caso, "salida_evaluacion_clasificacion".
El parámetro "per_device_eval_batch_size=" define el tamaño de batch por dispositivo, en este caso, 2.
El parámetro "eval_strategy=" define la estrategia de evaluación, en este caso, "no" para no realizar evaluaciones durante el entrenamiento.

La variable "argumentos" almacena la configuración del proceso, incluyendo el directorio de salida, el tamaño de batch por dispositivo y la estrategia de evaluación.
Estos argumentos controlan el comportamiento del proceso de evaluación del modelo de texto."""

from transformers import TrainingArguments

argumentos = TrainingArguments(
    output_dir="salida_evaluacion_clasificacion",
    per_device_eval_batch_size=2,
    eval_strategy="no"
)

argumentos # Muestra la configuración de los argumentos de entrenamiento.


In [ ]:
# Bloque 6 — Definición del Trainer
"""Este bloque define el Trainer que coordina el proceso de evaluación en clasificación.
Este bloque importa "Trainer" de "transformers" para coordinar el proceso de evaluación del modelo de texto.

Aplicamos la clase "Trainer()" a la variable "entrenador" para coordinar el proceso de evaluación del modelo.
El parámetro "model=" especifica el modelo que se evaluará, en este caso, "modelo".
El parámetro "args=" recibe los argumentos de evaluación, en este caso, "argumentos".
El parámetro "eval_dataset=" define el dataset de evaluación, en este caso, "dataset_eval_tokenizado".
El parámetro "compute_metrics=" recibe la función que calculará las métricas de evaluación, en este caso, "calcular_metricas".

La variable "entrenador" almacena el objeto Trainer configurado con el modelo, los argumentos y el dataset de evaluación.
Trainer gestiona automáticamente la ejecución de predicciones y el cálculo de métricas definidas."""

from transformers import Trainer      

entrenador = Trainer(
    model=modelo,
    args=argumentos,      
    eval_dataset=dataset_eval_tokenizado,
    compute_metrics=calcular_metricas
)

entrenador # Muestra el objeto Trainer configurado.


In [ ]:
# Bloque 7 — Ejecución de la evaluación
"""Este bloque ejecuta el proceso de evaluación mediante el método evaluate.
Aplicamos el método ".evaluate()" a la variable "entrenador" para iniciar el proceso de evaluación del modelo.
La variable "resultado_evaluacion" almacena las métricas generadas durante la evaluación del modelo.
Aunque "eval_strategy" sea "no", esta llamada explícita a ".evaluate()" sí ejecuta la evaluación."""

resultado_evaluacion = entrenador.evaluate()

resultado_evaluacion # Muestra las métricas generadas durante la evaluación del modelo.

In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del proceso de evaluación en clasificación.
La variable "resultado" almacena en forma de diccionario los resultados de la evaluación, incluidas las métricas de accuracy, precision, recall y F1."""

resultado = {
    "metricas": resultado_evaluacion
}

resultado # Muestra el diccionario con las métricas finales de evaluación.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto cómo aterrizar la evaluación general en una tarea clásica de clasificación de texto.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Entender el reparto de errores suele ser más útil que mirar una sola cifra de rendimiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Confiar en un resumen global sin revisar cómo responde el modelo en distintas clases o casos problemáticos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Análisis de Errores para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Análisis de Errores y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">07_Evaluacion_de_modelos_de_texto/04_El_analisis_de_errores.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">